# Assignment 1: ML Basics

**Name:** Tse Yan Wing Alvina  
**Team Members (if any):**

---

## Instructions
1. **Save your work:** In Colab, use **File → Save a copy in Drive**, or download the notebook to your local machine.
2. **Run in order:** Run cells from top to bottom because execution order matters.
3. **Complete the student areas:** Follow the comments in each section.
4. **Required experiments:** Document one baseline model plus at least five different hyperparameter configurations (at least six model runs total).
5. **Required metrics:** Report accuracy and the F1 score for the `ATTACK` class for every run. Use Attack F1 as the primary metric when selecting the best configuration.

---

***Attribution**: This notebook was created by Michael Roman for use in the CYBERSEC 520 course at Duke University. It was adapted from lecture notes. Gemini 3 Pro was used to format and clean up this notebook.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mproman/CYBERSEC520/blob/main/content/assignments/Assignment_1.ipynb)

## Step 0: Notebook Setup
We need to bring in our tools. These are the standard libraries used by Data Scientists everywhere.
* **Pandas:** For handling data like Excel sheets.
* **Seaborn/Matplotlib:** For drawing graphs.
* **Sklearn:** The Machine Learning library.

**Action:** Click the "Play" button on the cell below.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import machine-learning tools
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn import datasets
from sklearn.svm import SVC

# Make the graphs look nice
plt.style.use("seaborn-v0_8-whitegrid")
print("Setup complete! You are ready to start.")

Setup complete! You are ready to start.


## Step 1: Testing with Penguins
Before we tackle a Cyber dataset, let's make sure everything works with a simple dataset: **The Palmer Penguins**.

**Action:** Run the cell below to load the data.

In [2]:
# Load the data from a URL
url = "https://github.com/allisonhorst/palmerpenguins/raw/5b5891f01b52ae26ad8cb9755ec93672f49328a8/data/penguins_size.csv"
penguins = pd.read_csv(url)

# Drop missing values to prevent errors
penguins = penguins.dropna()

# Use two numerical features to predict the species
X_warmup = penguins[["culmen_length_mm", "flipper_length_mm"]]
y_warmup = penguins["species_short"]

# Split into training and test sets while preserving class proportions
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_warmup,
    y_warmup,
    test_size=0.2,
    random_state=42,
    stratify=y_warmup,
)

print("Penguin data loaded and split successfully.")
X_warmup.head()

Penguin data loaded and split successfully.


,culmen_length_mm,flipper_length_mm
0,39.1,181.0
1,39.5,186.0
2,40.3,195.0
4,36.7,193.0
5,39.3,190.0


### 1.1 Train a Simple Model
Now train either a **K-Nearest Neighbors (k-NN)** or **Support Vector Machine Classifier (SVC)** model on the warm-up data.

Use this section to confirm that you can create a model, fit it, generate predictions, and calculate a score before moving to the cybersecurity data.

In [3]:
# 1. Create the model
model = KNeighborsClassifier()

# 2. Train the model (The "Learning" part)
model.fit(X_train_w, y_train_w)

# 3. Predict on the test set
y_model = model.predict(X_test_w)

# 4. Check the score
accuracy_score(y_test_w, y_model)


0.9701492537313433

## Step 2: Cybersecurity Data
Now that the warm-up works, use the assignment data.

**Instructions:**
1. Download `Monday-WorkingHours.pcap_ISCX.csv` and one attack-day CSV from the [Canvas Datasets folder](https://canvas.duke.edu/courses/82545/files/folder/Datasets).
2. Combine the Monday file with the attack-day file. **Do not use Monday by itself:** it contains only benign traffic.
3. Create a binary target with two classes: `BENIGN` and `ATTACK`.
4. Clean missing, infinite, and invalid values before modeling.
5. Subsample the data to reduce runtime.
6. Update both path variables in the loading cell below.

> **Sampling hint:** A simple random sample may omit rare attacks. Consider sampling from `BENIGN` and `ATTACK` separately. When splitting the sample, use a fixed random seed and `stratify=y` so both classes appear in the training and test sets.

The Google Drive cell is optional and is needed only when your files are stored in Drive while using Colab.

In [4]:
# Optional: mount Google Drive when running this notebook in Colab.
# Skip this cell if you uploaded files directly, use Kaggle, or run locally.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Google Drive mounting is only needed in Colab.")

Mounted at /content/drive


In [6]:
# Set the paths to the two CSV files.
benign_path = "/content/drive/MyDrive/CYBERSEC520/Datasets/Monday-WorkingHours.pcap_ISCX.csv" # Monday-WorkingHours.pcap_ISCX.csv
attack_path = "/content/drive/MyDrive/CYBERSEC520/Datasets/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"  # One attack-day CSV

# Example Colab path:
# benign_path = "/content/drive/MyDrive/CYBERSEC520/Datasets/Monday-WorkingHours.pcap_ISCX.csv"

if benign_path is None or attack_path is None:
    raise ValueError("Set both benign_path and attack_path before running this cell.")

# Load and combine the benign and attack-day datasets.
df_benign = pd.read_csv(benign_path)
df_attack = pd.read_csv(attack_path)
df = pd.concat([df_benign, df_attack], ignore_index=True)

print(f"Combined size: {len(df):,} rows")

Combined size: 755,663 rows


### 2.1 Clean, Label, and Subsample
Cybersecurity datasets are large and often contain missing or infinite values.

In the next cell:

1. Normalize the column names.
2. Remove invalid values.
3. Convert the original labels into binary `BENIGN` and `ATTACK` labels.
4. Create a smaller sample for faster experiments.
5. Print the class counts and verify that both classes are present.

**Hint:** Sample from each class separately instead of taking one unrestricted random sample. The comments provide guidance, but you must complete the implementation.

In [13]:
# Complete each step below.

# 1. Normalize column names so " Label" and "Label" are handled consistently.
df.columns = df.columns.str.strip()

# 2. Replace positive/negative infinity with NaN, then remove invalid rows.
df.replace([-np.inf, np.inf], np.nan, inplace = True)
df = df.dropna()

# 3. Create a binary target named "BinaryLabel".
# Keep BENIGN as "BENIGN" and map every other original label to "ATTACK".
df["BinaryLabel"] = np.where(df["Label"] == "BENIGN", "BENIGN", "ATTACK")

# 4. Subsample the data.
# HINT: A single random sample may omit rare attacks. Consider sampling
# separately from each BinaryLabel group, using the same random_state.
#
# One possible structure is:
pieces = []
for label, group in df.groupby("BinaryLabel"):
     pieces.append(group.sample(n=min(1000, len(group)), random_state=42))
df_small = pd.concat(pieces, ignore_index=True)
df_small = df_small.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Verify the result before continuing.
# Print the original size, sampled size, and BinaryLabel value counts.
# Confirm that both BENIGN and ATTACK are present.
print(len(df))
print(len(df_small))
df_small["BinaryLabel"].value_counts()

755192
2000


,count
BinaryLabel,
BENIGN,1000
ATTACK,1000


### 2.2 Prepare the Data
Separate the **features** from the binary **target**, and then create training and test sets.

Use `BinaryLabel` as the target. Do not include either the original `Label` column or `BinaryLabel` among the model features.

Use the same sampled data, preprocessing, split, and random seed for the baseline and all five hyperparameter configurations.

In [18]:
# Complete each step below.

# 1. Inspect the available columns and class counts.
df_small["BinaryLabel"].value_counts()
print(df.head())

# 2. Define the target.
y = df_small["BinaryLabel"]

# 3. Define the feature matrix.
# Drop both the original "Label" column and "BinaryLabel" from X.
X = df_small.drop(columns = ['Label', 'BinaryLabel'])


# 4. Create one fixed stratified train/test split.
X_train, X_test, y_train, y_test = train_test_split(
     X,
     y,
     test_size=0.2,
     random_state=42,
     stratify=y,
 )

# 5. Optional but recommended for k-NN and SVM: scale numerical features.
# Fit StandardScaler only on X_train, then transform X_train and X_test.
# Use exactly the same prepared data for the baseline and all experiments.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0             49188              4                  2                       0   
1             49188              1                  2                       0   
2             49188              1                  2                       0   
3             49188              1                  2                       0   
4             49486              3                  2                       0   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                           12                            0   
1                           12                            0   
2                           12                            0   
3                           12                            0   
4                           12                            0   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0                      6                      6            

## Step 3: Baseline Model
The baseline is your chosen classifier using its **default hyperparameters**, such as `KNeighborsClassifier()` or `SVC()`.

Keep the dataset, preprocessing, train/test split, and random seed unchanged for every later experiment. Report both accuracy and Attack F1. Use Attack F1 as the primary metric when determining whether a configuration improves upon the baseline.

**Action:** Complete the student area below.

In [19]:
# --- STUDENT AREA ---

# 1. Initialize your chosen model with default hyperparameters.
# Example model classes: KNeighborsClassifier or SVC
clf_baseline = KNeighborsClassifier()

# 2. Fit the model on X_train and y_train.
clf_baseline.fit(X_train, y_train)

# 3. Predict on X_test.
y_pred_base = clf_baseline.predict(X_test)

# --- END STUDENT AREA ---

# Calculate the required baseline metrics.
baseline_accuracy = accuracy_score(y_test, y_pred_base)
baseline_attack_f1 = f1_score(y_test, y_pred_base, pos_label="ATTACK")

print(f"Baseline accuracy: {baseline_accuracy:.4f}")
print(f"Baseline Attack F1: {baseline_attack_f1:.4f}")

Baseline accuracy: 0.9600
Baseline Attack F1: 0.9604


## Step 4: Hyperparameter Exploration
Train and evaluate at least **five different hyperparameter configurations** in addition to the baseline. Your notebook must therefore contain at least **six model runs total**.

For k-NN, possible hyperparameters include:

- `n_neighbors`
- `weights`
- `metric`

For SVM, possible hyperparameters include:

- `C`
- `kernel`
- `gamma`
- `degree`

The `gamma` parameter applies to the `rbf`, `poly`, and `sigmoid` kernels.

For every configuration:

1. Train the model using the same training data and preprocessing.
2. Predict on the same test data.
3. Record accuracy and Attack F1.
4. Compare the result with the baseline.

Duplicate the experiment cell as needed. Additional experiments are encouraged.

In [20]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 1"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 3)

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 1
Accuracy: 0.9600
Attack F1: 0.9604


In [21]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 2"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 7)

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 2
Accuracy: 0.9600
Attack F1: 0.9606


In [27]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 3"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 8)

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 3
Accuracy: 0.9575
Attack F1: 0.9582


In [28]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 4"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 11)

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 4
Accuracy: 0.9500
Attack F1: 0.9505


In [29]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 5"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 15)

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 5
Accuracy: 0.9350
Attack F1: 0.9366


After testing with n = 3, 7, 8, 11 and 15 nearest neighbors for the algorithm, the accuracy of the testing stays stable for n <= 7 but starts to decay as n increases. This is because the model starts to look at general, large scale trends as larger n and overlooks tinier details in the data points in smaller areas, so the model actually becomes less complex. The Attack F1 score also decreases as n increases due to a similar reason. It seems like n = 7 is the sweet (optiimal) spot, not too small to caus overfitting, but not too large such that the model becomes too simple.

In [30]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 6"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(weights = "distance")
#closer neighbors have stronger influence

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 6
Accuracy: 0.9600
Attack F1: 0.9604


In [39]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 7"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 7, weights = "distance")
#closer neighbors have stronger influence

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 7
Accuracy: 0.9625
Attack F1: 0.9630


In [40]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 8"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 8, weights = "distance")
#closer neighbors have stronger influence

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 8
Accuracy: 0.9575
Attack F1: 0.9578


In [41]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 9"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 11, weights = "distance")
#closer neighbors have stronger influence

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 9
Accuracy: 0.9525
Attack F1: 0.9531


I tried changing the way the weights were weighted to "distance". It seems like the results do not really change if we keep n = 5, but the results improve for n = 7. Thinking that the accuracy and attack F1 would improve for all n, I tested n = 8 and n = 11 as well. However, the results worsen for n = 8, while results improve for n = 11. I think the performance might really depend on the data (like whether there are huge outliers for certain n, if so, "distance" would probably work better than "uniform"). I cannot find a general trend as of yet.

In [43]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 10"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(metric = "cosine")
#closer neighbors have stronger influence

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 10
Accuracy: 0.9725
Attack F1: 0.9730


In [44]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 11"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 7, metric = "cosine")
#closer neighbors have stronger influence

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 11
Accuracy: 0.9700
Attack F1: 0.9706


In [52]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 12"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 11, metric = "cosine")
#closer neighbors have stronger influence

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 12
Accuracy: 0.9600
Attack F1: 0.9612


In [53]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 13"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 3, metric = "cosine")
#closer neighbors have stronger influence

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 13
Accuracy: 0.9750
Attack F1: 0.9754


In [55]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.

experiment_name = "Configuration 14"

# 1. Initialize the same model type with different hyperparameters.
clf_experiment = KNeighborsClassifier(n_neighbors = 3, weights = "distance", metric = "cosine")
#closer neighbors have stronger influence

# 2. Fit the model using the same training data and preprocessing.
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
y_pred_experiment = clf_experiment.predict(X_test)

# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.

Configuration 14
Accuracy: 0.9825
Attack F1: 0.9826


I tried using the cosine metric instead of the default L2 norm. Testing n = 5, 7 and 11, the accuracy and attack f1 score seemed to show improvement compared to using the L2 norm. The results seemed to improve as n decreased, so I tried n = 3. As expected, the results improved for n = 3. Remembering that the way the distance was weighted also affected the results, I tried the combination n = 3, weighted = distance and cosine metric. It gave the best results as of yet. The cosine metric might be better in this case because we have quite a lot of parameters for each data point and we have to work in a higher-dimensional space.

## Step 5: Documentation and Reflection
*Double-click this cell to edit it and answer every question.*

### Experiment summary

| Experiment | Hyperparameters | Accuracy | Attack F1 |
|---|---|---:|---:|
| Baseline | Default settings |0.9600|0.9604|
| Configuration 1 |n_neighbors = 3 |0.9600|0.9604|
| Configuration 2 |n_neighbors = 7|0.9600|0.9606|
| Configuration 3 |n_neighbors = 8|0.9575|0.9582|
| Configuration 4 |n_neighbors = 11|0.9500|0.9505|
| Configuration 5 |n_neighbors = 15|0.9350|0.9366|
| Configuration 6 |weights = "distance"|0.9600|0.9604|
| Configuration 7 |n_neighbors = 7, weights = "distance"|0.9625|0.9630|
| Configuration 8 |n_neighbors = 8, weights = "distance"|0.9575|0.9578|
| Configuration 9 |n_neighbors = 11, weights = "distance"|0.9525|0.9531|
| Configuration 10 |metric = "cosine"|0.9725|0.9730|
| Configuration 11 |n_neighbors = 7, metric = "cosine"|0.9700|0.9706|
| Configuration 12 |n_neighbors = 11, metric = "cosine"|0.9600|0.9612|
| Configuration 13 |n_neighbors = 3, metric = "cosine"|0.9750|0.9754|
| Configuration 14 |n_neighbors = 3, weights = "distance", metric = "cosine"|0.9825|0.9826|

### Reflection questions

1. **What dataset and model did you choose?**
2. **How did you combine, clean, and subsample the data?**
3. **How many `BENIGN` and `ATTACK` observations were in your final sample?**
4. **What were the baseline accuracy and Attack F1?**
5. **Which hyperparameters and configurations did you test?**
6. **Which configuration performed best according to Attack F1?**
7. **How much did the best configuration improve over the baseline?**
8. **Why do you think that configuration performed better?**
9. **Did accuracy and Attack F1 lead to the same conclusion? Why or why not?**
10. **What problems or errors did you encounter, and how did you resolve them?**
11. **If you had more time, what would you try next?**

Evidence of thoughtful exploration is where the bulk of the assignment grade comes from.

1. I chose the "Monday-WorkingHours.pcap_ISCX.csv" for the benign dataset, and the "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv" for the attack dataset. I used the K-nearest neighbors model.

2. I did that using the toolkits and instructions provided.

3. There were 1000 BENIGN and 1000 ATTACK
observations.

4. The baseline accuracy was 0.9600, and the baseline Attack F1 was 0.9604.

5. I varied the number of neighbors n, the way the data was weighted, and the metric used to calculate the distance between points. The configuration table can be seen above.

6. n_neighbors = 3, weights = "distance", metric = "cosine"

7. The accuracy went from 0.9600 to 0.9825 (0.0225 increase), while the attack F1 went from 0.9604 to 0.9826 (0.0222 increase).

8. It should be because of a combination of things: as the data is high-dimensional, the cosine metric generally worked better, and the structure of the data itself really matters when it comes to which n works better and how to weigh the distances (like are there any outliers? Does the data have a general pattern/shape?)

9. Yes, it led to the same conclusion. The dataset is balanced, with the same nuber of BENIGN and ATTACK datapoints.

10. I am quite new to machine learning and libraries like pandas, np etc. in general, so it was a lot of trying to understand the syntax / what the code was supposed to do. I figured it out by reading the syntax documentation and searching online for resources.

11. I would like to test out different models and maybe see which model performs better on the data given, or this type of data in general. Also, I would like to learn how to perform data processing on other attacks, maybe not just DDos.

## Submission Checklist

- [V] I included my name.
- [V] I listed all team members, if any.
- [V] I combined the Monday benign dataset with one attack-day dataset.
- [V] My final sample contains both `BENIGN` and `ATTACK` observations.
- [V] I used one fixed train/test split for every experiment.
- [V] I documented one default-parameter baseline and at least five hyperparameter configurations.
- [V] I reported accuracy and Attack F1 for every run.
- [V] I completed every reflection question.
- [V] I ran all completed cells in order and preserved their outputs.
- [V] I documented generative-AI use according to the syllabus.

**To submit:** In Colab, select **File → Download → Download .ipynb**. Submit the completed notebook and, if your documentation is separate, the separate write-up through Canvas.